In [ ]:
import os, sys

SPACE = "YOUR_USERNAME/forge-ma"   # ← PUT YOUR HF USERNAME HERE

os.system(f"git clone https://huggingface.co/spaces/{SPACE} /content/forge-ma")
sys.path.insert(0, "/content/forge-ma")
os.chdir("/content/forge-ma")

# FIXED: correct Colab unsloth install (plain pip install fails)
os.system('pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"')

# FIXED: openenv-core was completely missing
os.system('pip install -q "openenv-core[core]>=0.2.1"')

os.system("pip install -q trl transformers torch torch-geometric sentence-transformers accelerate peft")
print("Setup complete.")

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("Model loaded.")

In [ ]:
# ── Cell 3: Red Team reward function ──────────────────────────
import numpy as np
from env.forge_env import ForgeEnv, ForgeEnvConfig
from env.primitives import PrimitiveType
from blue_team.gin_predictor import GINPredictor

blue_gin = GINPredictor()
PRIMITIVE_VALUES = [p.value for p in PrimitiveType]  # e.g. ["SOURCE_LAUNDER", ...]

def _extract_primitives(text: str) -> list:
    """Extract declared primitives from LLM output. Looks for PRIMITIVES: line."""
    found = []
    upper = text.upper()
    # First try explicit PRIMITIVES: line
    for line in upper.splitlines():
        if line.strip().startswith("PRIMITIVES:"):
            declared = line.split(":", 1)[1]
            for pname in PRIMITIVE_VALUES:
                if pname in declared:
                    found.append(pname)
            return found[:4]  # K_MAX
    # Fallback: scan whole text
    for pname in PRIMITIVE_VALUES:
        if pname in upper:
            found.append(pname)
    return found[:4]

def _safe_step(env, action):
    """
    FIXED: env.step() always requires an action argument.
    Returns (reward, done) regardless of return type.
    """
    result = env.step(action)
    if isinstance(result, tuple):
        if len(result) == 5:
            _, reward, terminated, truncated, _ = result
            return float(reward), bool(terminated or truncated)
        elif len(result) == 2:
            _, reward = result
            return float(reward), False
    return float(getattr(result, "reward", 0.0)), bool(getattr(result, "done", False))

def _safe_reset(env):
    """Returns (obs, info) regardless of implementation."""
    result = env.reset()
    if isinstance(result, tuple) and len(result) == 2:
        return result
    return result, {}

def _blue_predict_chain(env) -> list:
    """
    FIXED: GINPredictor uses predict_chain(g) not predict(graph).
    Returns a dict — access ["ordered_chain"] not .predicted_chain.
    """
    graph = getattr(env, "_claim_graph", None)
    if graph is None:
        return []
    try:
        result = blue_gin.predict_chain(graph)  # FIXED method name
        return result.get("ordered_chain", [])   # FIXED: dict not attribute
    except Exception:
        return []

def _tactic_edit_distance(pred: list, true: list) -> float:
    """
    FIXED: rewards.tactic_edit_dist module does not exist in codebase.
    Implement TED inline using normalised Levenshtein on string lists.
    """
    if not true:
        return 0.0
    if not pred:
        return 1.0
    # Normalised edit distance
    n, m = len(true), len(pred)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): dp[i][0] = i
    for j in range(m + 1): dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if true[i-1] == pred[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
    return dp[n][m] / max(n, m)

def reward_fn(completions, **kwargs):
    """
    Red Team GRPO reward — FULLY FIXED.

    Design fix: Red LLM declares which primitives it embedded.
    We treat those declared primitives as the "true chain".
    Blue GIN predicts from the generated env graph.
    red_reward = 1 - TED(blue_predicted, declared)

    Higher reward = blue failed to detect what red claimed to embed.
    """
    rewards = []
    for completion in completions:
        # FIXED: completion[-1]["content"] gets model output, not [0] which is user prompt
        if isinstance(completion, list):
            text = completion[-1].get("content", "")
        else:
            text = str(completion)

        declared = _extract_primitives(text)

        if not declared:
            rewards.append(0.001)
            continue

        try:
            cfg = ForgeEnvConfig(budget=5, seed=None)
            env = ForgeEnv(cfg)
            _safe_reset(env)

            # Run env steps — FIXED: step() requires action argument
            done = False
            for step_i in range(cfg.budget):
                if done:
                    break
                # Use tool actions to build up a claim graph for GIN to evaluate
                _, done = _safe_step(env, step_i % 8)

            blue_predicted_raw = _blue_predict_chain(env)
            # Normalise blue predictions to string values
            blue_strs = [
                p.value if hasattr(p, "value") else str(p)
                for p in blue_predicted_raw
            ]
            ted = _tactic_edit_distance(blue_strs, declared)
            red_reward = float(1.0 - ted)
            rewards.append(max(0.001, min(0.999, red_reward)))

        except Exception as e:
            print(f"Red reward error: {e}")
            rewards.append(0.001)

    return rewards

print("Red Team reward_fn ready.")

In [ ]:
from datasets import Dataset
import random

RED_PROMPT = """You are the Red Team Agent. Your goal is to embed manipulation primitives into a claim.
Declare the primitives you are embedding using the format:
PRIMITIVES: [primitive1, primitive2, ...]

Choose up to 4 primitives from:
SOURCE_LAUNDER, EMOTIONAL_APPEAL, OUT_OF_CONTEXT, SYNTHETIC_IMAGE, LOGICAL_FALLACY
"""

N = 100
prompts = [[{"role": "user", "content": RED_PROMPT}] for _ in range(N)]
dataset = Dataset.from_dict({"prompt": prompts})
print(f"Red Dataset size: {len(dataset)}")

In [ ]:
# ── Cell 5: Baseline evaluation ────────────────────────────────
import numpy as np, random

def evaluate_red_baseline(n_episodes=20):
    """
    FIXED: all env.step() calls now have action argument.
    FIXED: GIN uses predict_chain() not predict().
    FIXED: tactic_edit_distance implemented inline.
    FIXED: true_chain from env state, not info dict (which doesn't have it).
    """
    rewards = []
    for _ in range(n_episodes):
        try:
            cfg = ForgeEnvConfig(budget=5, seed=random.randint(0, 9999))
            env = ForgeEnv(cfg)
            _, info = _safe_reset(env)   # FIXED: unpack correctly

            done = False
            for step_i in range(cfg.budget):
                if done:
                    break
                _, done = _safe_step(env, step_i % 8)   # FIXED: action required

            # Get true chain from env directly (not from info dict)
            true_chain_raw = getattr(env, "_true_chain", [])
            true_strs = [
                p.value if hasattr(p, "value") else str(p)
                for p in true_chain_raw
            ]

            # FIXED: predict_chain() not predict(); returns dict not object
            blue_predicted_raw = _blue_predict_chain(env)
            blue_strs = [
                p.value if hasattr(p, "value") else str(p)
                for p in blue_predicted_raw
            ]

            # FIXED: inline TED, no import from non-existent module
            ted = _tactic_edit_distance(blue_strs, true_strs)
            rewards.append(max(0.001, min(0.999, 1.0 - ted)))

        except Exception as e:
            print(f"Baseline error: {e}")
            rewards.append(0.5)

    return float(np.mean(rewards))

baseline_red_reward = evaluate_red_baseline()
print(f"RED TEAM BASELINE — Mean reward (1 - TED): {baseline_red_reward:.4f}")
print("Higher = Blue fails more to detect manipulation = Red wins more")

In [ ]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir="./forge-red-grpo",
    max_steps=150,              # FIXED: explicit max_steps for T4 time limit
    num_generations=4,
    max_completion_length=256,
    per_device_train_batch_size=1,    # FIXED: was 2 → OOM on T4 with 3B model
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    save_steps=50,
    logging_steps=5,
    report_to="none",
    warmup_ratio=0.1,
)

# FIXED: processing_class for TRL>=0.9, fallback for older
try:
    trainer = GRPOTrainer(
        model=model,
        reward_funcs=reward_fn,
        args=config,
        train_dataset=dataset,
        processing_class=tokenizer,
    )
except TypeError:
    trainer = GRPOTrainer(
        model=model,
        reward_funcs=reward_fn,
        args=config,
        train_dataset=dataset,
        tokenizer=tokenizer,
    )

print("Starting RED TEAM GRPO training...")
trainer.train()
print("Red Team training complete.")

In [ ]:
import matplotlib.pyplot as plt, matplotlib, os
matplotlib.use('Agg')
os.makedirs("results", exist_ok=True)

log_history = trainer.state.log_history
# FIXED: try multiple key variants (TRL version differences)
steps, red_rewards = [], []
for l in log_history:
    for key in ("rewards/mean", "reward/mean", "reward"):
        if key in l:
            steps.append(l["step"])
            red_rewards.append(l[key])
            break

trained_red_reward = evaluate_red_baseline(n_episodes=20)
print(f"RED TEAM TRAINED  — Mean reward: {trained_red_reward:.4f}")
print(f"IMPROVEMENT: {trained_red_reward - baseline_red_reward:+.4f}")
print(f"Blue GIN detection rate: {1-baseline_red_reward:.1%} → {1-trained_red_reward:.1%}")

fig, ax = plt.subplots(figsize=(10, 5))
if steps:
    ax.plot(steps, red_rewards, color='#e74c3c', linewidth=2.5, label='Red Team reward')
ax.axhline(y=baseline_red_reward, color='gray', linestyle='--',
           linewidth=2, label=f'Untrained baseline ({baseline_red_reward:.3f})')
ax.set_xlabel("Training Step", fontsize=13)
ax.set_ylabel("Mean Red Reward (1 - Blue TED)", fontsize=13)
ax.set_title("FORGE-MA: Red Team LLM Learning Adversarial Deception via GRPO",
             fontsize=14, fontweight='bold')
ax.legend(fontsize=12); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig("results/red_reward_curve.png", dpi=150, bbox_inches='tight')
plt.close()

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(['Untrained Red', 'GRPO Trained Red'],
              [baseline_red_reward, trained_red_reward],
              color=['#95a5a6', '#e74c3c'], edgecolor='black', width=0.5)
ax.set_ylabel("Mean Red Reward (1 - Blue TED)", fontsize=13)
ax.set_title("Red Team: Before vs After GRPO\n(FORGE-MA Adversarial Agent)", fontsize=13)
ax.set_ylim(0, 1.0)
for bar, val in zip(bars, [baseline_red_reward, trained_red_reward]):
    ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("results/red_before_after.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved: results/red_reward_curve.png + results/red_before_after.png")
print("COMMIT BOTH TO REPO BEFORE SUBMITTING")